# Text preprocessingClean the tweet text, resolve label noise, and produce a stratified train/validationsplit. Cleaning comes from `src/text_cleaner.py` so that **train and test receiveidentical treatment** — applying it to only one split produces a good validationscore and a bad leaderboard score.Tokenization is deliberately *not* done here: it belongs after the split, fittedon the training half only.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.text_cleaner import clean_text

CLEAN = PROJECT_ROOT / "DATA" / "Clean"
PROCESSED = PROJECT_ROOT / "DATA" / "Processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

## Cleaning mode`"lstm"` — lowercase, strip punctuation and digits. For a from-scratch `Embedding`layer with a small vocabulary.`"transformer"` — keep casing, punctuation and digits, which WordPiece/BPE use assignal. Only URLs, @mentions, mojibake and HTML entities are removed.Both keep stopwords: the target architectures are sequence models that depend onword order.

In [2]:
MODE = "lstm"

In [3]:
train = pd.read_csv(CLEAN / "cleaned_train_data.csv")
test = pd.read_csv(CLEAN / "cleaned_test_data.csv")
print(train.shape, test.shape)

(7613, 5) (3263, 4)


In [4]:
# Both frames, same function, same mode.
train["text"] = train["text"].apply(clean_text, mode=MODE)
test["text"] = test["text"].apply(clean_text, mode=MODE)

print("test rows still containing a URL:", int(test['text'].str.contains('http').sum()))
train[["text", "target"]].head()

test rows still containing a URL: 0


,text,target
0,our deeds are the reason of this earthquake ma...,1
1,forest fire near la ronge sask canada,1
2,all residents asked to shelter in place are be...,1
3,people receive wildfires evacuation orders in ...,1
4,just got sent this photo from ruby alaska as s...,1


In [5]:
# Confirm the artifacts EDA found are gone from both frames.
import re

both = pd.concat([train["text"], test["text"]])
tokens = pd.Series([w for s in both for w in s.split()])
print("mojibake tokens :", int(tokens.str.contains(r"[\x80-\x9fÛåÌ]", regex=True).sum()))
print("'amp' tokens    :", int((tokens == "amp").sum()))
print("rows w/ URL     :", int(both.str.contains("http").sum()))
print("rows cleaned to empty:", int((both.str.strip() == "").sum()))
print("vocabulary size :", tokens.nunique())

mojibake tokens : 0
'amp' tokens    : 2
rows w/ URL     : 0
rows cleaned to empty: 0
vocabulary size : 17381


## Duplicates and conflicting labelsDeduplication runs *after* cleaning so that near-duplicates differing only by URLor casing collapse together.Some duplicate groups disagree on the label. `keep="first"` would resolve those byCSV row order, i.e. arbitrarily. Instead: take the majority label, and drop groupsthat tie — a 50/50 group carries no recoverable signal and would only add noise toboth train and validation.

In [6]:
dupes = train[train.duplicated(subset=["text"], keep=False)]
groups = dupes.groupby("text")["target"].nunique()
print("duplicate rows      :", len(dupes))
print("duplicate groups    :", len(groups))
print("conflicting groups  :", int((groups > 1).sum()))
print("rows in conflict    :", int(dupes[dupes['text'].isin(groups[groups > 1].index)].shape[0]))

duplicate rows      : 1166
duplicate groups    : 357
conflicting groups  : 86
rows in conflict    : 319


In [7]:
def resolve_labels(df):
    """Collapse duplicate texts to one row: majority label, ties dropped."""
    counts = df.groupby(["text", "target"]).size().rename("n").reset_index()
    top = counts.sort_values("n", ascending=False).groupby("text").head(1)

    # A group ties when its best label is not strictly more common than the rest.
    totals = counts.groupby("text")["n"].sum().rename("total")
    top = top.join(totals, on="text")
    decided = top[top["n"] * 2 > top["total"]]

    n_tied = top.shape[0] - decided.shape[0]
    print(f"dropped {n_tied} tied groups")
    return decided[["text", "target"]]

labels = resolve_labels(train)

# One row per unique text, carrying the resolved label.
meta = train.drop(columns=["target"]).drop_duplicates(subset=["text"], keep="first")
train = labels.merge(meta, on="text", how="left")
train = train[train["text"].str.strip() != ""]

print("rows after dedup + resolution:", len(train))
print(train["target"].value_counts(normalize=True).round(4))

dropped 40 tied groups
rows after dedup + resolution: 6764
target
0    0.5931
1    0.4069
Name: proportion, dtype: float64


## Sequence length

In [8]:
lengths = train["text"].str.split().str.len()
print(lengths.describe().round(2))
print("\npercentiles:", lengths.quantile([.90, .95, .99, 1.0]).round(1).to_dict())

MAXLEN = int(lengths.quantile(0.99))
print("\nMAXLEN =", MAXLEN, "-> covers %.1f%% of tweets uncut"
      % ((lengths <= MAXLEN).mean() * 100))

count    6764.00
mean       13.64
std         5.91
min         1.00
25%         9.00
50%        13.00
75%        18.00
max        32.00
Name: text, dtype: float64

percentiles: {0.9: 22.0, 0.95: 24.0, 0.99: 27.0, 1.0: 32.0}

MAXLEN = 27 -> covers 99.5% of tweets uncut


## Stratified split`X_val` is the held-out slice of the *training* data. It is not the Kaggle testset — that stays in `test` and has no labels.The split happens before tokenization: fit the tokenizer on `X_train` only, orthe validation score leaks vocabulary it should not have seen.

In [9]:
from sklearn.model_selection import train_test_split

X = train.drop(columns=["target"])
y = train["target"]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("train:", X_train.shape, "val:", X_val.shape, "kaggle test:", test.shape)
print("train balance:", y_train.value_counts(normalize=True).round(4).to_dict())
print("val   balance:", y_val.value_counts(normalize=True).round(4).to_dict())

train: (5411, 4) val: (1353, 4) kaggle test: (3263, 4)
train balance: {0: 0.5931, 1: 0.4069}
val   balance: {0: 0.5935, 1: 0.4065}


## Save

In [10]:
X_train.assign(target=y_train).to_csv(PROCESSED / f"train_{MODE}.csv", index=False)
X_val.assign(target=y_val).to_csv(PROCESSED / f"val_{MODE}.csv", index=False)
test.to_csv(PROCESSED / f"test_{MODE}.csv", index=False)

for p in sorted(PROCESSED.glob(f"*_{MODE}.csv")):
    print(f"{p.name:18} {len(pd.read_csv(p)):5} rows")

test_lstm.csv       3263 rows
train_lstm.csv      5411 rows
val_lstm.csv        1353 rows


### NextFit `Tokenizer(num_words=..., oov_token="<OOV>")` on `X_train["text"]` only, thentransform and pad all three splits to `MAXLEN`. Design the architecture after that.